# Khai báo các hàm phụ trợ tương tự như xử lí Dataset

In [42]:
import os
import pandas as pd
import numpy as np
import cv2
from matplotlib import pyplot as plt
import mediapipe as mp

## Trích xuất keyframes

In [43]:
def getGrayFramesAndFrames(frame_buffer):
    gray_frames = [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for frame in frame_buffer]
    return gray_frames

In [44]:
def calculate_histogram_differences(gray_frames):
    HDiffs = []
    # Duyệt qua tất cả các frame trừ frame cuối vì thường là frame không có giá trị
    for i in range(0, len(gray_frames)-1):
        if (i == 0):
        #[gray_frames[i]]: grayframe thứ i, [0] kênh chứa độ sáng, None: tính toàn bộ ảnh, không dùng mask, [256]: 256 bins, 0 <=[0, 256]: giá trị pixel < 256
            hist_curr = cv2.calcHist([gray_frames[i]], [0], None, [256], [0, 256])
            continue
        # Gán frame ở vòng lặp trước cho hist_prev và tính lại hist_curr
        hist_prev = hist_curr
        hist_curr = cv2.calcHist([gray_frames[i]], [0], None, [256], [0, 256])

        Hdiff = np.sum(np.abs(hist_prev - hist_curr))
        HDiffs.append(Hdiff)
    return HDiffs

In [45]:
def Extract_key_frames(frames):
    gray_frames = getGrayFramesAndFrames(frames)
    HDiffs = calculate_histogram_differences(gray_frames)
    mean = np.mean(HDiffs)
    std = np.std(HDiffs)
    threshold = mean + std
    # Chọn keyframes dựa trên ngưỡng
    keyframes = []
    for i in range(len(HDiffs)):
        if HDiffs[i] > threshold:
            # Lấy frame i+1 vì Hdiffs[i] là độ khác biệt của frame thứ i +1 với thứ i
            keyframes.append(frames[i+1])
    return keyframes

## Trích xuất landmarks

In [46]:
def euclidean_distance(v1, v2):
    return np.sqrt((float(v1[0]) - float(v2[0])) ** 2 + (float(v1[1]) - float(v2[1])) ** 2)

In [47]:
def extract_pose_landmarks(rgb_frame, mp_pose):
    pose_results = mp_pose.process(rgb_frame)
    pose_landmarks = []

    if pose_results.pose_landmarks:
        for i, lm in enumerate(pose_results.pose_landmarks.landmark):
            if i < 17 and i not in [7, 8]:  # Loại bỏ từ hông trở xuống và 2 tai
                pose_landmarks.append((lm.x, lm.y))

    return pose_landmarks

In [48]:
def classify_hands (pose_landmarks, hand_landmarks):
    left_wrist_pose  = pose_landmarks[13]  # Cổ tay trái từ Pose là 15 trừ đi 2 tai đã lượt bỏ nên idx = 13
    right_wrist_pose = pose_landmarks[14]  # Cổ tay phải từ Pose là 16 trừ đi 2 tai đã lượt bỏ nên idx = 14
    wrist = hand_landmarks[0] 
    

    dleft = euclidean_distance(left_wrist_pose, wrist)
    dright = euclidean_distance(right_wrist_pose, wrist)
   
    if(dleft < dright):
        return "Left"
    else:
        return "Right"

In [49]:
def extract_hand_landmarks(rgb_frame, mp_hands, pose_landmarks):
    hands_results = mp_hands.process(rgb_frame)
    left_hand_landmarks = []
    right_hand_landmarks = []
    

    if hands_results.multi_hand_landmarks and hands_results.multi_handedness:
        for hand_landmarks, handedness in zip(hands_results.multi_hand_landmarks, hands_results.multi_handedness):
            # Không sử dụng hướng tay của mediapipe vì độ chính xác thấp và thường ngược hướng
            landmarks = [(lm.x, lm.y) for lm in hand_landmarks.landmark]
            if classify_hands(pose_landmarks, landmarks) == "Right":
                right_hand_landmarks = landmarks
            else:
                left_hand_landmarks = landmarks   



        
    
    return left_hand_landmarks, right_hand_landmarks

In [50]:
def extract_landmarks(frames):
    mp_pose = mp.solutions.pose.Pose(static_image_mode=True)
    mp_hands = mp.solutions.hands.Hands(static_image_mode=True, max_num_hands=2, min_detection_confidence=0.1)
    
    landmarks_dict = {}
    
    for idx, frame in enumerate(frames):
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        
        pose_landmarks = extract_pose_landmarks(rgb_frame, mp_pose)
        left_hand_landmarks, right_hand_landmarks = extract_hand_landmarks(rgb_frame, mp_hands ,pose_landmarks)
        
        landmarks_dict[idx] = {
            "pose": pose_landmarks,
            "left": left_hand_landmarks,
            "right": right_hand_landmarks
        }
    #giải phóng tài nguyên
    mp_pose.close()
    mp_hands.close()
    return landmarks_dict

In [51]:
def filter_invalid_landmarks(landmarks_dict):
    validated = {}
    for landmarks_idx, landmarks_data in landmarks_dict.items():
        validated_landmarks = {}
        # Nếu không có pose thì bỏ qua frame
        if  not landmarks_data["pose"]:
            continue
        for part in ["pose", "right", "left"]:
            # Nếu không có dữ liệu cho phần này, gán mặc định
            if part not in landmarks_data or not landmarks_data[part]:
                if part in ["right", "left"]:
                    validated_landmarks[part] = [(0.0, 0.0)] * 21
            else:
                processed_points = []
                for point in landmarks_data[part]:
                    x = float(point[0])
                    y = float(point[1])
                    if not (0.0 <= x <= 1.0 and 0.0 <= y <= 1.0):
                        x, y = 0.0, 0.0
                    processed_points.append((x, y))
                validated_landmarks[part] = processed_points
        validated[landmarks_idx] = validated_landmarks
    return validated

## Chuẩn hóa với sign space

In [52]:
def calculate_head_unit(pose_landmarks):

    left_eye, right_eye = pose_landmarks[3], pose_landmarks[6]
    # mép ngoài 2 mắt
    head_unit = euclidean_distance(left_eye,right_eye)
    return head_unit


In [53]:
def calculate_sign_space(pose_landmarks):
    head_unit = calculate_head_unit(pose_landmarks)

    nose = pose_landmarks[0]
   
   
    width = 7 * head_unit
    # height = 9.5 * head_unit 
    
    center_x, center_y = nose

    x1 = center_x - width / 2

    y1 = center_y - 1.5 * head_unit  # Cạnh trên cách 1,5 head unit
    x2 = center_x + width / 2
    # y2 = min(1.0 , int(center_y + 7.5 * head_unit))
    y2 = center_y + 8 * head_unit 
    return [x1, y1, x2, y2]

In [54]:
def calculate_all_sign_space(landmarks_dict):
    sign_spaces = {}
    for idx, landmarks_data in landmarks_dict.items():
        sign_spaces[idx] = calculate_sign_space(landmarks_data["pose"])
    return sign_spaces

In [55]:
def normalize_landmarks_to_sign_space(landmarks_dict, sign_spaces):
    
    normalized = {}
    for landmarks_idx, landmarks_data in landmarks_dict.items():
        Xmin, Ymin, Xmax, Ymax = sign_spaces[landmarks_idx]
        w = Xmax - Xmin
        h = Ymax - Ymin
        normalized_landmarks = {}
        for part in ["pose", "right", "left"]:
            processed_points = []
            for point in landmarks_data[part]:
                x = float(point[0])
                y = float(point[1])
                if x != 0.0 and y != 0.0:
                   x = (x - Xmin) / w
                   y = (y - Ymin) / h
                processed_points.append((x, y))
            normalized_landmarks[part] = processed_points
        normalized[landmarks_idx] = normalized_landmarks
    return normalized

# Xử lí từ livecam

In [56]:
import cv2
import mediapipe as mp
import time
import threading
from queue import Queue

In [57]:
# Khởi tạo pose từ MediaPipe
mp_pose = mp.solutions.pose
pose = mp_pose.Pose()
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

# Queue để model xử lý
action_queue = Queue()

# Cấu hình timeout
MAX_IDLE_FRAMES = 5
MIN_IDLE_FRAMES = 5

In [58]:
# === XỬ LÝ TRẠNG THÁI TAY ===
def check_action_state(pose_landmarks):
    left_wrist = pose_landmarks.landmark[15]
    right_wrist = pose_landmarks.landmark[16]
    left_hip = pose_landmarks.landmark[23]
    right_hip = pose_landmarks.landmark[24]

    hips_y = min(left_hip.y, right_hip.y)

    # Nếu cả 2 tay thấp hơn hông hoặc nằm ngoài khung hình
    if ((left_wrist.y * 1.1 > hips_y or left_wrist.y > 1) and
        (right_wrist.y * 1.1 > hips_y or right_wrist.y > 1)):
        return "IDLE"
    
    # Nếu 1 trong 2 tay cao hơn hông và trong khung hình
    if ((left_wrist.y < hips_y and left_wrist.y < 1) or
        (right_wrist.y < hips_y and right_wrist.y < 1)):
        return "ACTIVE"
    
    return "IDLE"

In [59]:
# === THREAD XỬ LÝ MODEL ===
def model_worker():
    while True:
        segment = action_queue.get()
        if segment is None:
            break
        print(f"[MODEL] Nhận đoạn {len(segment)} frames để xử lý.")
        keyframes = Extract_key_frames(segment)
        landmarks_dict = extract_landmarks(keyframes)
        filtered_landmarks = filter_invalid_landmarks(landmarks_dict)
        sign_spaces = calculate_all_sign_space(filtered_landmarks)
        normalized_landmarks = normalize_landmarks_to_sign_space(filtered_landmarks, sign_spaces)
        print(normalized_landmarks)
        # TODO: Gọi model tại đây để nhận diện thủ ngữ
        action_queue.task_done()

# Khởi chạy thread xử lý model
threading.Thread(target=model_worker, daemon=True).start()


[MODEL] Nhận đoạn 86 frames để xử lý.
{0: {'pose': [(0.5, 0.15789473684210523), (0.5193089532877531, 0.09851186901926197), (0.5401798756877013, 0.09101132491044739), (0.5585071946853939, 0.08475460536771441), (0.4612354635338213, 0.11941089401226115), (0.4442605411376218, 0.1250259060332671), (0.429831670270148, 0.13047904332065202), (0.5547142092665245, 0.18570965807902848), (0.4815514343176652, 0.21300999805130305), (0.8035285122476139, 0.2916988251235712), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.823255450917917, 0.05544758834569546), (0.0, 0.0)], 'right': [(0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0), (0.0, 0.0)], 'left': [(0.8071237792418463, 0.03550675734042078), (0.7334136547538157, 0.03460728448040573), (0.6668487739145561, -0.007887064213144918), (0.6277927815496943, -0.05407842295197594)

In [60]:
# === XỬ LÝ CAMERA ===
cap = cv2.VideoCapture(0)

frame_buffer = []
idle_count = 0
is_action_active = False

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)

    if results.pose_landmarks:
        state = check_action_state(results.pose_landmarks)

        # ACTIVE → đang thực hiện động tác
        if state == "ACTIVE":
            frame_buffer.append(frame.copy())
            idle_count = 0
            if not is_action_active:
                print("[INFO] → BẮT ĐẦU động tác")
                is_action_active = True

        # IDLE → nghỉ tay
        elif state == "IDLE":
            if is_action_active:
                idle_count += 1
                frame_buffer.append(frame.copy())

                if idle_count >= MAX_IDLE_FRAMES:
                    print("[INFO] → KẾT THÚC động tác, đẩy vào model")

                    # Cắt 5 frame IDLE cuối
                    valid_segment = frame_buffer[:-MAX_IDLE_FRAMES] if len(frame_buffer) > MAX_IDLE_FRAMES else []

                    if valid_segment:
                        action_queue.put(valid_segment)
                        print(f"[DEBUG] → Đoạn động tác: {len(valid_segment)} frames.")

                    # Reset
                    frame_buffer.clear()
                    idle_count = 0
                    is_action_active = False

        # Vẽ trạng thái lên màn hình
        cv2.putText(frame, f"State: {state}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0) if state == "ACTIVE" else (0, 0, 255), 2)
    mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            landmark_drawing_spec=mp_drawing_styles.get_default_pose_landmarks_style())
    cv2.imshow("Sign Language Capture", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break



[INFO] → BẮT ĐẦU động tác
[INFO] → KẾT THÚC động tác, đẩy vào model
[DEBUG] → Đoạn động tác: 86 frames.


KeyboardInterrupt: 

In [61]:
# Dọn dẹp
cap.release()
cv2.destroyAllWindows()
action_queue.put(None)  # Dừng thread